In [ ]:
# 데이터셋은 sklearn wine 데이터셋 사용
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

- 정규 방정식을 활용한 선형 회귀 구현하기

In [ ]:
# 정규 방정식(Normal Equation) 방식으로 변경된 Linear Regression

# Linear_regression 직접 구현하기
    # 정규 방정식을 사용하므로 학습률과 반복 횟수가 필요 없음
# 1. 정규 방정식 활용
# 2. fit 메서드 정의
    # 해석적으로 최적해를 직접 계산
# 3. predict 메서드 정의
    # 계산된 가중치와 편향으로 예측
class LinearRegression:
    def __init__(self):
        # 정규 방정식은 학습률과 반복 횟수가 필요 없음
        self.weights = None  # 가중치 초기화
        self.bias = None    # 편향 초기화

    def fit(self, X, y):
        # 샘플 개수, 특성 개수
        n_samples, n_features = X.shape     # (142, 13)
        
        # 정규 방정식을 위해 X에 편향(bias) 항을 추가
        # X_with_bias = [1, x1, x2, ..., xn] 형태로 만들어 편향을 가중치에 포함
        X_with_bias = np.column_stack([np.ones(n_samples), X])
        
        # 정규 방정식: θ = (X^T * X)^(-1) * X^T * y
        # θ는 [bias, weight1, weight2, ..., weightn]을 포함하는 벡터
        try:
            # X_with_bias.T @ X_with_bias: 공분산 행렬 계산
            # np.linalg.inv(): 역행렬 계산
            # X_with_bias.T @ y: X^T와 y의 행렬곱
            theta = np.linalg.inv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
            
            # theta의 첫 번째 원소는 편향, 나머지는 가중치
            self.bias = theta[0]
            self.weights = theta[1:]
            
        except np.linalg.LinAlgError:
            # 역행렬이 존재하지 않는 경우 의사역행렬(pseudo-inverse) 사용
            # 이는 특이값 분해(SVD)를 사용한 안정적인 방법
            theta = np.linalg.pinv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
            self.bias = theta[0]
            self.weights = theta[1:]

    def predict(self, X):
        # 계산된 가중치와 편향으로 예측
        return X @ self.weights + self.bias
    
# 모델 학습 및 평가
reg_normal = LinearRegression()

# 모델 학습 (정규 방정식은 한 번의 계산으로 최적해를 구함)
reg_normal.fit(X_train, y_train)

# 테스트 데이터로 예측
y_pred = reg_normal.predict(X_test)

- 성능 평가

In [ ]:
results = pd.DataFrame({'예측 등급': y_pred, '실제 등급': y_test.values})
print(results.head())

# 모델 성능 평가

# 평균 제곱 오차(MSE) 계산
# 0에 가까울수록 좋은 모델
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

mse = mean_squared_error(y_test, y_pred)
print(mse)

# 결정 계수(R²) 계산
# 1에 가까울수록 좋은 모델
def r2_score(y_true, y_pred):
    ss_total = np.sum((y_true - np.mean(y_true))**2)  # 총 변동
    ss_residual = np.sum((y_true - y_pred)**2)        # 잔차 제곱합
    return 1 - (ss_residual / ss_total)               # 결정 계수

r2 = r2_score(y_test, y_pred)
print(r2)

- 선형회귀 표준화 직접 구현하기

In [ ]:
X, y = load_wine(return_X_y=True, as_frame=True)
# print(X.head, y.values)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# print(X_train.shape, y_train.shape)

# 와인데이터는 특성간 편차가 매우 크기 때문에 스케일링 과정 필요
class StandardScaler:
    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.scale_ = np.std(X, axis=0)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train_scaled[:5].to_numpy())


- 경사 하강법을 활용한 선형 회귀 직접 구현하기

In [ ]:
# Linear_regression 직접 구현하기
    # 학습률, 학습할 횟수 정보 필요
# 1. 경사 하강법 활용
# 2. fit 메서드 정의
    # 정해진 횟수만큼 경사하강법 반복
# 3. predict 메서드 정의
    # 최종 학습된 가중치와 편향으로 예측
class LinearRregression:
    # 학습률 기본 값: 0.01
    # 반복 횟수 기본 값: 1000
    def __init__(self, lr=0.001, n_iter=1000):
        # 학습률(learning rate): 가중치를 얼마나 크게 업데이트 할지 결정
        self.lr = lr
        # 경사 하강법을 반복할 횟수
        self.n_iter = n_iter
        self.weights = None  # 가중치 초기화
        self.bias = None    # 편향 초기화

    def fit(self, X, y):
        # 샘플 개수, 특성 개수
        n_samples, n_features = X.shape     # (142, 13)
        # 가중치와 편향을 0으로 초기화
        # 각 특성별 계산이 필요하므로, 특성의 개수의 크기에 맞게 0으로 채운 벡터
        self.weights = np.zeros(n_features)
        self.bias = 0

        # 경사 하강법 반복
        for _ in range(self.n_iter):
            # 예측값 계산: y_pred = X @ w + b
                # @: 행렬 곱
            y_pred = X.dot(self.weights) + self.bias

            # 오차 계산
            # error = 오차 벡터 (142,)
            error = y_pred - y

            '''가중치와 편향의 경사 계산 (경사 계산 == 변화율을 계산)            
            가중치
                X.T @ error
                    각 특성이 오차에 얼마나 기여했는지를 계산
                    만약, 어떤 특성의 값이 클 때 오차가 크다면,
                    그 특성의 가중치를 더 많이 조정해야 한다는 것을 의미
                (1/n_samples)
                    전체 샘플수로 나누어 평균을 구할 것
                    데이터의 크기에 관계 없이 경사의 평균 값을 계산
            
            편향
                - 가중치와 마찬가지로 평균을 구해서 처리
            '''            
              
            dw = (1/n_samples) * X.T @ error
            db = (1/n_samples) * np.sum(error)

            # 가중치와 편향 업데이트
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        # 최종 학습된 가중치와 편향으로 예측
        return X @ self.weights + self.bias
    
# 모델 학습 및 평가
reg_gd = LinearRregression(lr=0.1, n_iter=100)

# 모델 학습
reg_gd.fit(X_train_scaled, y_train)

# 테스트 데이터로 예측
y_pred = reg_gd.predict(X_test_scaled)


- 성능 평가

In [ ]:
results = pd.DataFrame({'예측 등급': y_pred, '실제 등급': y_test.values})
print(results.head())

# 모델 성능 평가

# 평균 제곱 오차(MSE) 계산
# 0에 가까울수록 좋은 모델
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

mse = mean_squared_error(y_test, y_pred)
print(mse)

# 결정 계수(R²) 계산
# 1에 가까울수록 좋은 모델
def r2_score(y_true, y_pred):
    ss_total = np.sum((y_true - np.mean(y_true))**2)  # 총 변동
    ss_residual = np.sum((y_true - y_pred)**2)        # 잔차 제곱합
    return 1 - (ss_residual / ss_total)               # 결정 계수

r2 = r2_score(y_test, y_pred)
print(r2)